### what this model will do 
we will use NLP to check a given transaction description like "swiggy order" or "monthly house rent" 
the model will then predict the category like "food" and "rent"

pipeline would be something like:
raw test -> clean text -> TF-IDF numbers -> model -> categpry

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import joblib

In [3]:
## improting data
df = pd.read_excel("../data/sample_transactions.xlsx")
df.head()

,transaction_id,date,day_of_week,month,description,merchant,amount (₹),category,payment_mode,transaction_channel,account_type,city,is_weekend,is_anomaly
0,TXN00001,2024-01-02,Tuesday,January,POS TXN 9289,Cafe Coffee Day,8123.00,Food,Cash,ATM,Savings,Delhi,False,True
1,TXN00002,2024-01-02,Tuesday,January,dunzo delivery,Dunzo,315.93,Food,NEFT,Web,Savings,Delhi,False,False
2,TXN00003,2024-01-02,Tuesday,January,property tax payment,BBMP,4619.46,Utilities,Debit Card,Auto Debit,Savings,Delhi,False,False
3,TXN00004,2024-01-02,Tuesday,January,monthly rent,Housing Society,14768.18,Rent,Net Banking,Auto Debit,Savings,Delhi,False,False
4,TXN00005,2024-01-02,Tuesday,January,monthly rent,Housing Society,20437.67,Rent,Auto Debit,Auto Debit,Savings,Delhi,False,False


In [4]:
## changing data from string to date format
df['date'] = pd.to_datetime(df['date'])

print(df['date'].head)
print(df['date'].dtype)

<bound method NDFrame.head of 0      2024-01-02
1      2024-01-02
2      2024-01-02
3      2024-01-02
4      2024-01-02
          ...    
1393   2024-12-30
1394   2024-12-30
1395   2024-12-30
1396   2024-12-31
1397   2024-12-31
Name: date, Length: 1398, dtype: datetime64[us]>
datetime64[us]


### what we are doing
train a model that predicts the spending category from the text description of the transaction

eg. input "Swiggy order" -> output "food"

In [5]:
## observing the data
print(df[['description', 'category']].head(10))
print("\nUnique categories are: ", df["category"].nunique())
print(df["category"].value_counts())

            description   category
0          POS TXN 9289       Food
1        dunzo delivery       Food
2  property tax payment  Utilities
3          monthly rent       Rent
4          monthly rent       Rent
5          Swiggy order       Food
6             SWG ORDER       Food
7  PG accommodation fee       Rent
8   society maintenance       Rent
9  PG accommodation fee       Rent

Unique categories are:  15
category
Food             278
Groceries        182
Shopping         155
Entertainment    101
Travel           101
Bills             88
Rent              75
Utilities         74
Investments       61
Healthcare        61
Fuel              61
Education         47
Personal Care     40
Fitness           40
Insurance         34
Name: count, dtype: int64


In [6]:
## data cleaning
df['clean_description'] = df['description'].str.lower().str.strip()

print(df[['description','clean_description']].head(10))

            description     clean_description
0          POS TXN 9289          pos txn 9289
1        dunzo delivery        dunzo delivery
2  property tax payment  property tax payment
3          monthly rent          monthly rent
4          monthly rent          monthly rent
5          Swiggy order          swiggy order
6             SWG ORDER             swg order
7  PG accommodation fee  pg accommodation fee
8   society maintenance   society maintenance
9  PG accommodation fee  pg accommodation fee


In [7]:
## making x-input and y-output

X = df['clean_description']
y = df['category']

print("Feature shape: ", X.shape)
print("target shape: ",y.shape)

Feature shape:  (1398,)
target shape:  (1398,)


In [8]:
## train test split

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

print("training sample length ", len(X_train))
print("testing sample length ", len(X_test))

training sample length  1118
testing sample length  280


In [9]:
# Vectorization using TF-IDF

vectorizer =  TfidfVectorizer(ngram_range=(1,2))

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("vocabulary size: ", len(vectorizer.vocabulary_)) ## this will tell the number of unique words
print("training matrix shape: ", X_train_tfidf.shape)

vocabulary size:  780
training matrix shape:  (1118, 780)


In [10]:
## training the model
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_tfidf, y_train)

print("random forst model trained")

random forst model trained


In [11]:
model2 = LogisticRegression(max_iter=1000)
model2.fit(X_train_tfidf,y_train)
print("LogisticRegression model trained")

LogisticRegression model trained


In [12]:
## evaluating the performance of random forest

y_pred = model.predict(X_test_tfidf)

print("Random forest classifier \n")
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("\n", classification_report(y_test, y_pred))

Random forest classifier 

Accuracy:  0.875

                precision    recall  f1-score   support

        Bills       0.67      0.80      0.73        15
    Education       1.00      0.67      0.80         6
Entertainment       0.92      1.00      0.96        22
      Fitness       1.00      0.50      0.67         4
         Food       0.82      1.00      0.90        47
         Fuel       0.75      0.94      0.83        16
    Groceries       1.00      0.91      0.95        34
   Healthcare       0.71      0.83      0.77        12
    Insurance       0.60      0.86      0.71         7
  Investments       1.00      0.60      0.75        15
Personal Care       1.00      0.75      0.86         8
         Rent       1.00      0.86      0.92        14
     Shopping       1.00      0.79      0.89        39
       Travel       0.89      0.96      0.92        25
    Utilities       0.93      0.88      0.90        16

     accuracy                           0.88       280
    macro avg    

In [13]:
## evaluating the performance of logistic regression model

y_pred = model2.predict(X_test_tfidf)

print("LogisticRegression")
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("\n", classification_report(y_test, y_pred))

LogisticRegression
Accuracy:  0.8714285714285714

                precision    recall  f1-score   support

        Bills       0.71      0.80      0.75        15
    Education       0.80      0.67      0.73         6
Entertainment       0.92      1.00      0.96        22
      Fitness       1.00      0.50      0.67         4
         Food       0.82      1.00      0.90        47
         Fuel       1.00      0.75      0.86        16
    Groceries       0.87      1.00      0.93        34
   Healthcare       0.71      0.83      0.77        12
    Insurance       0.67      0.86      0.75         7
  Investments       1.00      0.53      0.70        15
Personal Care       1.00      0.75      0.86         8
         Rent       1.00      0.86      0.92        14
     Shopping       0.94      0.79      0.86        39
       Travel       0.89      0.96      0.92        25
    Utilities       0.93      0.88      0.90        16

     accuracy                           0.87       280
    macro av

## conclusion
- randomforestclassifier gives slightly better accuracy than logistic regression
- accuracy achieved is: 87.5%
- the model signifacntly struggled with predicting rent as it gave f1 score of 0.4 in earlier cases with precsion of only 0.25
- this problem was caused as the model got confused in the monthly payments of rent and other payments like insurance and investments
- we tackled it by changing the description of rents so the model clearly can differentiate between rent payments and inurance or investemnts

In [25]:
## testing the model 
def predict_category(description):
    cleaned = description.lower().strip()
    tfidf = vectorizer.transform([cleaned])
    prediction = model.predict(tfidf)
    return prediction[0]

print(predict_category('Swiggy Order1234'))
print(predict_category('swiigy'))
print(predict_category("Monthly house rent"))
print(predict_category("Petrol refill"))
print(predict_category("Doctor consultation"))

Food
Entertainment
Rent
Fuel
Healthcare


In [27]:
## saving the model

joblib.dump(model, "../models/categorizer_v1_description_only.pkl")
joblib.dump(vectorizer,"../models/tfidf_vectorizer_v1.pkl")
print("version 1 using only description feature is saved")

version 1 using only description feature is saved
